***Phase 1: Data Exploration***

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

df = pd.read_csv("../data/raw/global_ads_performance_dataset.csv", parse_dates=["date"])
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Shape: (1800, 14)
Columns: ['date', 'platform', 'campaign_type', 'industry', 'country', 'impressions', 'clicks', 'CTR', 'CPC', 'ad_spend', 'conversions', 'CPA', 'revenue', 'ROAS']


,date,platform,campaign_type,industry,country,impressions,clicks,CTR,CPC,ad_spend,conversions,CPA,revenue,ROAS
0,2024-01-21,Google Ads,Search,Fintech,UAE,59886,2113,0.0353,1.2600,"2,662.3800",159,16.7400,"4,803.4300",1.8000
1,2024-01-22,TikTok Ads,Search,EdTech,UK,135608,5220,0.0385,1.1800,"6,159.6000",411,14.9900,"64,126.6800",10.4100
2,2024-06-15,TikTok Ads,Video,Healthcare,USA,92313,5991,0.0649,0.8500,"5,092.3500",267,19.0700,"10,489.0700",2.0600
3,2024-01-02,TikTok Ads,Shopping,SaaS,Germany,83953,5935,0.0707,1.3200,"7,834.2000",296,26.4700,"50,505.0700",6.4500
4,2024-02-22,TikTok Ads,Search,Healthcare,UK,91807,4489,0.0489,1.9300,"8,663.7700",107,80.9700,"3,369.5300",0.3900


In [2]:
df.info()
print("\nNull counts:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           1800 non-null   datetime64[ns]
 1   platform       1800 non-null   object        
 2   campaign_type  1800 non-null   object        
 3   industry       1800 non-null   object        
 4   country        1800 non-null   object        
 5   impressions    1800 non-null   int64         
 6   clicks         1800 non-null   int64         
 7   CTR            1800 non-null   float64       
 8   CPC            1800 non-null   float64       
 9   ad_spend       1800 non-null   float64       
 10  conversions    1800 non-null   int64         
 11  CPA            1800 non-null   float64       
 12  revenue        1800 non-null   float64       
 13  ROAS           1800 non-null   float64       
dtypes: datetime64[ns](1), float64(6), int64(3), object(4)
memory usage: 197.

In [3]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
date,1800,2024-06-28 12:56:00,2024-01-01 00:00:00,2024-03-27 18:00:00,2024-06-26 00:00:00,2024-09-26 00:00:00,2024-12-30 00:00:00,NaN
impressions,"1,800.0000","102,919.0189","5,059.0000","54,948.0000","103,653.0000","150,470.2500","199,650.0000","55,740.9007"
clicks,"1,800.0000","3,962.6756",91.0000,"1,678.0000","3,318.0000","5,628.0000","16,660.0000","2,941.8580"
CTR,"1,800.0000",0.0384,0.0089,0.0254,0.0355,0.0498,0.0956,0.0171
CPC,"1,800.0000",1.5728,0.2800,0.9500,1.4600,2.0500,3.9500,0.8009
ad_spend,"1,800.0000","6,171.5273",58.0000,"1,966.5875","4,393.8600","8,455.8300","38,453.3200","5,776.9970"
conversions,"1,800.0000",181.5622,2.0000,59.0000,130.0000,252.2500,"1,151.0000",171.4242
CPA,"1,800.0000",46.6090,4.8000,20.2025,33.3750,56.8125,335.8600,41.1856
revenue,"1,800.0000","30,101.8505",142.6900,"7,275.7575","18,362.9650","38,963.3850","295,028.2600","34,560.0329"
ROAS,"1,800.0000",6.4504,0.1300,2.1700,4.2950,8.2125,49.0000,6.5910


In [4]:
cat_cols = list(df.select_dtypes(include="object").columns)
for col in cat_cols:
    print(f"--- {col} ({df[col].nunique()} unique) ---")
    print(df[col].value_counts(dropna=False).head(15), "\n")

print("Date range:", df["date"].min(), "->", df["date"].max())
print("Distinct dates:", df["date"].nunique())

--- platform (3 unique) ---
platform
Google Ads    720
Meta Ads      630
TikTok Ads    450
Name: count, dtype: int64 

--- campaign_type (4 unique) ---
campaign_type
Search      477
Video       456
Shopping    447
Display     420
Name: count, dtype: int64 

--- industry (5 unique) ---
industry
EdTech        372
SaaS          370
Fintech       361
E-commerce    349
Healthcare    348
Name: count, dtype: int64 

--- country (7 unique) ---
country
UK           266
USA          266
Canada       262
India        261
UAE          258
Germany      255
Australia    232
Name: count, dtype: int64 

Date range: 2024-01-01 00:00:00 -> 2024-12-30 00:00:00
Distinct dates: 360


In [5]:
checks = {
    "clicks > impressions": (df["clicks"] > df["impressions"]).sum(),
    "conversions > clicks": (df["conversions"] > df["clicks"]).sum(),
    "ad_spend <= 0": (df["ad_spend"] <= 0).sum(),
    "clicks == 0": (df["clicks"] == 0).sum(),
    "conversions == 0": (df["conversions"] == 0).sum(),
    "revenue < 0": (df["revenue"] < 0).sum(),
    "revenue == 0": (df["revenue"] == 0).sum(),
}
pd.Series(checks, name="row count").to_frame()

,row count
clicks > impressions,0
conversions > clicks,0
ad_spend <= 0,0
clicks == 0,0
conversions == 0,0
revenue < 0,0
revenue == 0,0


In [6]:
calc = pd.DataFrame({
    "CTR": df["clicks"] / df["impressions"],
    "CPC": df["ad_spend"] / df["clicks"],
    "CPA": df["ad_spend"] / df["conversions"],
    "ROAS": df["revenue"] / df["ad_spend"],
}).replace([np.inf, -np.inf], np.nan)

rows = []
for c in calc.columns:
    diff = (calc[c] - df[c]).abs()
    rows.append({
        "metric": c,
        "given_mean": df[c].mean(),
        "recomputed_mean": calc[c].mean(),
        "max_abs_diff": diff.max(),
        "mean_abs_diff": diff.mean(),
        "rows_diff_gt_0.01": int((diff > 0.01).sum()),
    })
pd.DataFrame(rows)

,metric,given_mean,recomputed_mean,max_abs_diff,mean_abs_diff,rows_diff_gt_0.01
0,CTR,0.0384,0.0384,0.0002,0.0000,0
1,CPC,1.5728,1.5728,0.0000,0.0000,0
2,CPA,46.6090,46.6089,0.0050,0.0025,0
3,ROAS,6.4504,6.4502,0.0050,0.0025,0


In [7]:
g = df.groupby("platform").agg(
    rows=("platform", "size"),
    spend=("ad_spend", "sum"),
    revenue=("revenue", "sum"),
    impressions=("impressions", "sum"),
    clicks=("clicks", "sum"),
    conversions=("conversions", "sum"),
)
g["CTR"] = g["clicks"] / g["impressions"]
g["CPC"] = g["spend"] / g["clicks"]
g["conv_rate"] = g["conversions"] / g["clicks"]
g["CAC"] = g["spend"] / g["conversions"]
g["ROAS"] = g["revenue"] / g["spend"]
g

,rows,spend,revenue,impressions,clicks,conversions,CTR,CPC,conv_rate,CAC,ROAS
platform,,,,,,,,,,,
Google Ads,720,"6,349,268.9100","22,033,744.9500",73801150,2936149,131098,0.0398,2.1624,0.0446,48.4315,3.4703
Meta Ads,630,"2,106,061.6700","11,926,045.7900",64403494,1596736,73262,0.0248,1.3190,0.0459,28.7470,5.6627
TikTok Ads,450,"2,653,418.5100","20,223,540.0700",47049590,2599931,122452,0.0553,1.0206,0.0471,21.6690,7.6217


**Findings:**
- Rows / columns: 1800 / 14
- Nulls: 0 | Duplicates: 0
- Logical inconsistencies: none (clicks > impressions, conversions > clicks, spend <= 0 all 0)
- Given vs recomputed metrics: rounding differences only (max 0.005), no scale issue
- Platform differences: clear (ROAS: TikTok 7.62, Meta 5.66, Google 3.47; CAC: 21.7 / 28.7 / 48.4)
- Date range: 2024-01-01 to 2024-12-30
- Note: mean of row-level ROAS (6.45) differs from aggregate ROAS (~4.88), so metrics must be computed from sums
